Implementing the Macro method from Couloumbe (2024)

Data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm

In [ ]:
df = pd.read_csv('england_master.csv')

In [ ]:
df['date'] = pd.to_datetime(df['Unnamed: 0'].str.replace('Q', '-Q'))
df.set_index('date', inplace=True)

In [ ]:
df_clean = df[['starts', 'hprice', 'cc', 'rate', 'vol']].copy()

In [ ]:
df_clean['starts_lag1'] = df_clean['starts'].shift(1)
df_clean['starts_lag4'] = df_clean['starts'].shift(4)
df_clean['vol_lag1'] = df_clean['vol'].shift(1)
df_clean['rate_lag1'] = df_clean['rate'].shift(1)
df_clean['time_trend'] = np.arange(len(df_clean))

In [ ]:
df_clean = df_clean.dropna()

In [ ]:
y = df_clean['starts']

In [ ]:
df_clean['starts'] = np.log(df_clean['starts'])
df_clean['hprice'] = np.log(df_clean['hprice'])
df_clean['cc'] = np.log(df_clean['cc'])

In [ ]:
X = df_clean[['hprice', 'cc', 'rate']]
X = sm.add_constant(X)

In [ ]:
S = df_clean[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]
print(f"Data cleaned. Proceeding with {len(y)} perfectly aligned quarters.")

The macroeconomic RF

In [ ]:
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf.fit(S, y)

In [ ]:
leaf_assignments = rf.apply(S)

In [ ]:
valid_idx = df_clean.index
gtvps = pd.DataFrame(index=valid_idx, columns=X.columns, dtype=float)
predicted_y = pd.Series(index=valid_idx, dtype=float)

In [ ]:
for t_idx in range(len(valid_idx)):
  current_leaves = leaf_assignments[t_idx, :]
  weights = np.sum(leaf_assignments == current_leaves, axis=1)
  weights = weights / weights.sum()
  wls_model = sm.WLS(y, X, weights=weights).fit()
  gtvps.iloc[t_idx] = wls_model.params
  predicted_y.iloc[t_idx] = np.dot(X.iloc[t_idx], wls_model.params)

In [ ]:
gtvps = gtvps.astype(float)

Visualisation


In [ ]:
#Plot1
plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['rate'], color='crimson', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of the BoE Base Rate on Housing Starts', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Rate)', fontsize=12)
plt.grid(True, alpha=0.3)

plt.fill_between(gtvps.index, gtvps['rate'], 0, where=(gtvps['rate'] < 0), color='crimson', alpha=0.1)

plt.tight_layout()
plt.savefig('gtvp_interest_rate.png')
plt.show()

In [ ]:
#plot2

plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['cc'], color='darkblue', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of Construction Costs', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Costs)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gtvp_construction_costs.png')
plt.show()